<a href="https://colab.research.google.com/github/Decuayer/ce477-data-science-term-project/blob/feature%2Fdata-pipeline/afad_earthquake_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys

# Ortam Colab mi kontrol et
IS_COLAB = "google.colab" in sys.modules
gc = None

if IS_COLAB:
    from google.auth import default
    from google.colab import auth, drive
    import gspread

    try:
        drive.mount("/content/drive")
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        print("Colab ortamı: Drive ve Sheets bağlandı.")
    except Exception as e:
        print(f"Colab yetkilendirme uyarısı: {e}")
else:
    print("Local ortam tespit edildi: Colab kütüphaneleri atlandı.")

Local ortam tespit edildi: Colab kütüphaneleri atlandı.


In [13]:
import os
from pathlib import Path
import pandas as pd

# 1. Dosya Yolu Belirleme (Local vs Colab)
if IS_COLAB:
    csv_cache_path = "/content/drive/MyDrive/afad_earthquake_data_raw.csv"
else:
    os.makedirs("./data", exist_ok=True)
    csv_cache_path = "./data/afad_earthquake_data_raw.csv"

df = None
sheet_title = "afad_earthquake_data_raw"

# 2. Öncelik 1: CSV Önbelleği (Localde ./data/, Colab'de Drive kontrolü)
if os.path.exists(csv_cache_path) and os.path.getsize(csv_cache_path) > 1000:
    source_name = "Google Drive (CSV)" if IS_COLAB else "Local Cache (CSV)"
    print(f"Loaded from {source_name}")
    df = pd.read_csv(csv_cache_path)

# 3. Öncelik 2: Google Sheets (Colab ortamında ve Sheets yetkisi varsa)
if df is None and IS_COLAB and gc is not None:
    try:
        print("Fetching from Google Sheets...")
        worksheet = gc.open(sheet_title).sheet1
        # Büyük verilerde get_all_records() yerine get_all_values() daha hızlı ve kararlıdır
        rows = worksheet.get_all_values()
        if len(rows) > 1:
            df = pd.DataFrame(rows[1:], columns=rows[0])
            print(f"Loaded from Google Sheets (Rows: {len(df)})")
            # Bir dahaki sefere hızlı açılması için Drive'a kaydet
            df.to_csv(csv_cache_path, index=False)
            print(f"Saved to Drive cache: {csv_cache_path}")
    except Exception as e:
        print(f"Google Sheets okuma hatası: {e}")

# 4. Durum Kontrolü
if df is not None and not df.empty:
    print(f"Shape: {df.shape}")
else:
    print(
        "Veri yüklenemedi! Local ortamdaysanız 'afad_earthquake_data_raw.csv' dosyasını './data/' klasörüne kopyalayınız."
    )

Veri yüklenemedi! Local ortamdaysanız 'afad_earthquake_data_raw.csv' dosyasını './data/' klasörüne kopyalayınız.


In [8]:
print(f"Shape: {df.shape}")
df.head()

Shape: (0, 15)


,Rms,EventID,Location,Latitude,Longitude,Depth,Type,Magnitude,Country,Province,District,Neighborhood,Date,isEventUpdate,lastUpdateDate
